# SAT-AI — flood model training (Colab / Kaggle)

Runs the whole flood pipeline end to end on a free GPU: clone, install,
download Sen1Floods11, train, evaluate on a held-out region, score the
classical baseline, and run the C2 ablation.

**Nothing here needs a secret.** The Sen1Floods11 bucket is public and the
training path never touches the Gemini key — that lives only on the serving
deployment. Do not paste any API key into this notebook.

Expect roughly 25 minutes per training run on a T4, and about 700 MB of
download. Runtime → Change runtime type → GPU before you start.

## 1. GPU check

Stop here if this reports no CUDA device. Training on CPU is possible and
takes hours, which is not what this notebook is for.

In [ ]:
import subprocess

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
        capture_output=True,
        text=True,
    ).stdout
    or "no nvidia-smi: switch the runtime to GPU"
)

## 2. Clone and install

Only the training dependencies. The geospatial stack the repository pins for
local work (GDAL, geopandas) is not needed here — rasterio alone reads the
chips.

In [ ]:
import os, sys, pathlib

REPO = pathlib.Path("SAT-AI")
if not REPO.exists():
    !git clone --depth 1 https://github.com/Sahil27-cs/SAT-AI.git

# The scripts resolve the repo root themselves, but setting this makes the
# lookup explicit and survives a cell being run from a different directory.
os.environ["SATAI_REPO_ROOT"] = str(REPO.resolve())
sys.path.insert(0, str(REPO.resolve()))

!pip install -q rasterio scipy scikit-learn pydantic pydantic-settings pyyaml

import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 3. Download Sen1Floods11

446 hand-labelled chips over 11 flood events: Sentinel-1 VV/VH plus the
hand-drawn water masks. Resumable — re-run the cell if it is interrupted.

In [ ]:
!cd SAT-AI && python scripts/download_sen1floods11.py --out data/raw/sen1floods11

## 4. Classical baseline

Run this **before** the deep model, not after. Otsu thresholding with a
unimodality guard is a genuinely strong operational method, and a U-Net that
cannot beat it has demonstrated nothing. Knowing the bar first is what stops
a mediocre deep result from looking like a good one.

In [ ]:
!cd SAT-AI && python -m ml.experiments.run_flood_baseline \n    --chips data/raw/sen1floods11 --min-separability 0.66

## 5. Train

Leave-one-region-out: India is the test region and never influences training
or checkpoint selection. Validation is a *different* held-out region, so early
stopping is not tuned on chips statistically identical to the training set.

Swap `--fold` for any of `loro_ghana`, `loro_mekong`, `loro_usa` and so on to
hold out a different region.

In [ ]:
!cd SAT-AI && python -m ml.flood.train \n    --data-root data/raw/sen1floods11 \n    --fold loro_india --epochs 40 --batch-size 8 --amp --patience 10

## 6. Evaluate on the test region

This is the number that may be reported. The training cell prints a
*validation* IoU on a different region; quoting that as the model's score
reports the region the checkpoint was selected on.

In [ ]:
!cd SAT-AI && python -m ml.flood.evaluate \n    --checkpoint models/flood/flood_unet_loro_india_sar_ratio_best.pt

## 7. Compare against the baseline

Both numbers from the same held-out region under the same protocol. The
reference run gave IoU 0.523 for the U-Net against 0.375 for Otsu.

In [ ]:
!cd SAT-AI && python -m ml.experiments.compare_flood_models

## 8. C2 — modality ablation

Trains a second arm without the derived VV/VH ratio band, holding seed,
schedule, fold and selection region fixed, then compares. In the reference
run the ratio band made no measurable difference (0.5211 vs 0.5230), which is
a negative result and is reported as one.

In [ ]:
!cd SAT-AI && python -m ml.flood.train \n    --data-root data/raw/sen1floods11 \n    --fold loro_india --epochs 40 --batch-size 8 --amp --patience 10 --no-ratio

!cd SAT-AI && python -m ml.flood.evaluate \n    --checkpoint models/flood/flood_unet_loro_india_sar_best.pt

!cd SAT-AI && python -m ml.experiments.run_modality_ablation

## 9. Keep the artifacts

Colab and Kaggle discard the filesystem when the session ends. The weights
are ~93 MB each; the JSON reports are small and are what the repository
actually tracks, since they are the evidence a number came from a real run.

In [ ]:
import shutil, pathlib

OUT = pathlib.Path("satai_artifacts")
OUT.mkdir(exist_ok=True)
for pattern in ("models/flood/*.json", "models/flood/*.pt", "ml/experiments/**/*.json"):
    for path in pathlib.Path("SAT-AI").glob(pattern):
        shutil.copy2(path, OUT / path.name)

print(f"{len(list(OUT.iterdir()))} artifacts in {OUT}/")
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:<52} {p.stat().st_size / 1024:8.1f} KB")

# On Colab, uncomment to pull them down:
# from google.colab import files
# shutil.make_archive('satai_artifacts', 'zip', OUT)
# files.download('satai_artifacts.zip')

## What this notebook does not do

- **C3 (rural → urban transfer)** needs labelled urban Indian flood imagery.
  Sen1Floods11's India chips are not urban, so the target does not exist in
  this dataset and the experiment stays unexecuted rather than approximated.
- **C2's rainfall and terrain arms** need GPM and Copernicus DEM rasters
  co-registered to each chip. Neither ships with Sen1Floods11.
- **C1 (Gemini grounding)** runs against the deployed serving plane, not here,
  and needs `GEMINI_API_KEY` set on that deployment — never in this notebook.